# Bayesian Spatial Models: A Pedagogical Walkthrough

This notebook demonstrates all five models currently implemented in `bayespecon`:

1. SLX
2. SAR
3. SEM
4. SDM
5. SDEM

Each section explains the model idea, fits the model, and inspects posterior summaries and spatial effects.


## Conceptual Roadmap

Let $W$ denote the spatial weights matrix and $WX$ the spatial lag of covariates.

* __SLX__: $y = X\beta + WX\theta + \epsilon$
* __SAR__: $y = \rho Wy + X\beta + \epsilon$
* __SEM__: $y = X\beta + u$, $u = \lambda Wu + \epsilon$
* __SDM__: $y = \rho Wy + X\beta + WX\theta + \epsilon$
* __SDEM__: $y = X\beta + WX\theta + u$, $u = \lambda Wu + \epsilon$

In `bayespecon`, all models accept either:

* formula mode: `model_class(formula=..., data=..., W=...)`, or
* matrix mode: `model_class(y=..., X=..., W=...)`.


In [ ]:
import arviz as az
import geodatasets
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from libpysal.graph import Graph

from bayespecon.diagnostics.bayesfactor import bayes_factor_compare_models
from bayespecon.models import OLS, SAR, SDEM, SDM, SEM, SLX

az.style.use("arviz-white")

In [ ]:
# Load sample spatial dataset
gdf = gpd.read_file(geodatasets.data.geoda.airbnb.url)
xcols = ["poverty", "rev_rating", "num_spots", "crowded"]
ycol = "price_pp"
gdf = gdf.dropna(subset=xcols + [ycol]).copy()

# Build contiguity graph (queen)
W = Graph.build_contiguity(gdf, rook=False).transform("r")

print(f"Observations: {len(gdf)}")
print(f"Predictors: {xcols}")

__Helper Functions__

To keep each model section focused, we use utility functions to:

* fit with consistent MCMC settings,
* print posterior summaries,
* tabulate direct/indirect/total effects.

For a quick pedagogical run, we use small draw counts. Increase these for real inference.


In [ ]:
def fit_and_report(model_cls, formula, data, W, draws=400, tune=400, chains=4, seed=42):
    """Fit a spatial model and return (model, summary, effects_df).

    Uses minimal MCMC settings for pedagogical demonstration.
    Increase draws/tune for real analyses.
    """
    model = model_cls(formula=formula, data=data, W=W, logdet_method="eigenvalue")
    model.fit(
        draws=draws,
        tune=tune,
        chains=chains,
        random_seed=seed,
        progressbar=False,
        idata_kwargs={"log_likelihood": True},
    )
    summary = model.summary(round_to=3)
    effects_df = pd.DataFrame(model.spatial_effects())
    return model, summary, effects_df

__Convergence Diagnostics Helper__

For each fitted model, we will inspect:

* `r_hat` (target close to 1.00)
* effective sample sizes (`ess_bulk`, `ess_tail`)
* trace plots for key parameters.


In [ ]:
def diagnostics_table(idata, var_names):
    """Show key MCMC diagnostics for the given parameters."""
    cols = ["mean", "sd", "ess_bulk", "ess_tail", "r_hat"]
    return az.summary(idata, var_names=var_names, round_to=3)[cols]


def show_trace(idata, var_names, title):
    """Plot trace plots for the given parameters."""
    az.plot_trace(idata, var_names=var_names)
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

## Models


### 0) OLS: What could go wrong?


Model:

$$
y = X\beta + \epsilon
$$


In [ ]:
ols = OLS(
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
olsfit = ols.fit(
    draws=400,
    tune=400,
    chains=4,
    random_seed=42,
    progressbar=False,
    idata_kwargs={"log_likelihood": True},
)

In [ ]:
ols.summary()

### 1) SLX: Spatially Lagged Covariates Only

Model:

$$
y = X\beta + WX\theta + \epsilon
$$

Interpretation:

* `beta` captures local covariate effects.
* `theta` captures neighbor-covariate spillovers.
* No spatial lag on $y$, so no autoregressive propagation through outcomes.


In [ ]:
slx, summary_slx, effects_slx = fit_and_report(
    SLX,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_slx)
display(effects_slx)

In [ ]:
az.plot_forest(slx.inference_data)
display(diagnostics_table(slx.inference_data, ["beta", "sigma"]))
show_trace(slx.inference_data, ["sigma"], "SLX Trace: sigma")

### 2) SAR: Spatial Lag of Outcome

Model:

$$
y = \rho Wy + X\beta + \epsilon
$$

Interpretation:

* $\rho$ controls feedback from neighboring outcomes.
* Effects are amplified through $(I - \rho W)^{-1}$, so direct and indirect impacts differ from raw $\beta$.


In [ ]:
sar, summary_sar, effects_sar = fit_and_report(
    SAR,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
summary_sar

In [ ]:
effects_sar

In [ ]:
az.plot_forest(sar.inference_data)
display(diagnostics_table(sar.inference_data, ["rho", "beta", "sigma"]))
show_trace(sar.inference_data, ["rho", "sigma"], "SAR Trace: rho, sigma")

### 3) SEM: Spatial Correlation in Errors

Model:

$$
y = X\beta + u, \quad u = \lambda Wu + \epsilon
$$

Interpretation:

* Spatial dependence is moved to latent shocks rather than outcome feedback.
* Useful when omitted spatial factors induce correlated residuals.


In [ ]:
sem, summary_sem, effects_sem = fit_and_report(
    SEM,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_sem)
display(effects_sem)

In [ ]:
az.plot_forest(sem.inference_data)
display(diagnostics_table(sem.inference_data, ["lam", "beta", "sigma"]))
show_trace(sem.inference_data, ["lam", "sigma"], "SEM Trace: lam, sigma")

### 4) SDM: SAR + SLX

Model:

$$
y = \rho Wy + X\beta + WX\theta + \epsilon
$$

Interpretation:

* Includes both outcome feedback and neighbor-covariate channels.
* Often used as a flexible nesting model for SAR/SLX.


In [ ]:
sdm, summary_sdm, effects_sdm = fit_and_report(
    SDM,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_sdm)
display(effects_sdm)

In [ ]:
az.plot_forest(sdm.inference_data)
display(diagnostics_table(sdm.inference_data, ["rho", "beta", "sigma"]))
show_trace(sdm.inference_data, ["rho", "sigma"], "SDM Trace: rho, sigma")

### 5) SDEM: SLX + Spatial Error

Model:

$$
y = X\beta + WX\theta + u, \quad u = \lambda Wu + \epsilon
$$

Interpretation:

* Neighbor covariates matter directly (through $WX$),
* and unmodeled shocks are spatially correlated (through $u$).


In [ ]:
sdem, summary_sdem, effects_sdem = fit_and_report(
    SDEM,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_sdem)
display(effects_sdem)

In [ ]:
az.plot_forest(sdem.inference_data)
display(diagnostics_table(sdem.inference_data, ["lam", "beta", "sigma"]))
show_trace(sdem.inference_data, ["lam", "sigma"], "SDEM Trace: lam, sigma")

### MCMC Sampling Adequacy

Bayesian estimation of spatial models has a well-known efficiency
pitfall: the spatial dependence parameter $\rho$ (or $\lambda$) often
mixes slowly, so a chain that _looks_ converged in its point estimate
can still produce posterior credible intervals that are 10–12 % too
narrow because the sampler has not visited the tails enough times
{cite:p}`wolf2018StochasticEfficiency`. The
`spatial_mcmc_diagnostic` helper checks effective sample size, sampler
yield, $\hat{R}$, and HPDI stability for the spatial parameter and
warns when any threshold is violated.


In [ ]:
from bayespecon.diagnostics import spatial_mcmc_diagnostic

# Run on the SDM (most parameters; both rho and impact summaries depend on it)
report = spatial_mcmc_diagnostic(sdm, emit_warnings=False)
report.to_frame()

In [ ]:
sdm.spatial_diagnostics()

In [ ]:
ols.spatial_diagnostics()

In [ ]:
sar.spatial_diagnostics_decision()

In [ ]:
sem.spatial_diagnostics()

In [ ]:
report

## Model Comparison


In [ ]:
# Collect all fitted models for comparison
model_dict = {
    "OLS": ols,
    "SLX": slx,
    "SAR": sar,
    "SEM": sem,
    "SDM": sdm,
    "SDEM": sdem,
}
idata_dict = {name: m.inference_data for name, m in model_dict.items()}

### Bayes Factor Model Comparison

Bayes factors provide an alternative to information criteria for comparing competing models. While WAIC and LOO assess _predictive_ performance, Bayes factors compare _marginal likelihoods_ — the probability of the data under each model, integrated over all parameter values weighted by the prior:

$$BF\_{ij} = \frac{p(y \mid \mathcal{M}\_i)}{p(y \mid \mathcal{M}\_j)} = \frac{ML\_i}{ML\_j}$$

This makes Bayes factors sensitive to the _prior_: models with diffuse priors on unnecessary parameters are penalized more heavily, because the marginal likelihood averages over all possible parameter values weighted by the prior. In spatial models, this means that models with many WX coefficients under wide priors (e.g., SLX, SDM, SDEM) can receive substantially lower marginal likelihoods than more parsimonious alternatives (e.g., OLS, SAR, SEM) — even if their log-likelihoods are similar.

__Interpreting Bayes factors__ (Kass & Raftery, 1995):

| BF range | Evidence strength |
| -------- | ----------------- |
| 1 – 3    | Anecdotal         |
| 3 – 10   | Moderate          |
| 10 – 30  | Strong            |
| 30 – 100 | Very strong       |
| > 100    | Extreme           |

__Method.__ We use bridge sampling (Meng & Wong, 1996) to estimate each model's log marginal likelihood, following the R `bridgesampling` package (Gronau et al., 2020). The implementation uses ESS weighting, two-phase convergence, and MCSE diagnostics. For reliable estimates, 40,000+ posterior samples are recommended.

__Important caveat.__ Bayes factors can be very sensitive to prior specification. Models with diffuse priors on many parameters will tend to have lower marginal likelihoods. This is by design — it is the Bayesian Occam's razor at work — but it means that Bayes factors and information criteria may disagree when priors are uninformative.


In [ ]:
bayes_factor_compare_models(model_dict, method="bridge", log=True).round(3)

In [ ]:
bayes_factor_compare_models(model_dict, method="bic", log=True).round(3)

#### Bridge Sampling vs. BIC

* BIC: SAR > SLX > OLS > SEM (SAR wins, SLX is moderate, SEM is worst)
* Bridge: SAR > SEM > OLS (SAR wins, SEM is moderate)

The two methods can produce qualitatively different model rankings. This is expected and reflects a fundamental difference in what they assume about priors:

* __BIC__ approximates $\log(ML) \approx \hat\ell\_{\max} - \frac{k}{2}\log(n)$, which assumes _unit-information priors_ (priors containing as much information as a single observation). The penalty per parameter is fixed at $\frac{1}{2}\log(n) \approx 2.1$ for $n = 77$.

* __Bridge sampling__ integrates over the _actual_ priors in the model. When priors are wide (e.g., `Normal(0, 100)` on WX coefficients), the marginal likelihood penalizes each such parameter by roughly $\log(\sigma\_{\text{prior}} / \sigma\_{\text{post}})$, which can be 5–10× larger than the BIC penalty.

This is why models with many WX terms (SLX, SDM, SDEM) may look reasonable under BIC but receive extreme Bayes factors under bridge sampling: the wide priors on the WX coefficients are "wasted" — they spread probability mass over implausible parameter values, reducing the marginal likelihood. This is __Bayesian Occam's razor__ at work, and bridge sampling is generally more trustworthy because it accounts for the actual prior specification.


## Bayesian LM Specification Tests

The `bayespecon.diagnostics` module provides Bayesian Lagrange-multiplier tests that operate on posterior draws rather than point estimates. These replace the frequentist LM tests that were previously available.

Below we run the Bayesian LM tests from the fitted OLS model:

* `bayesian_lm_lag_test` — tests $H\_0: \rho = 0$ (no spatial lag)
* `bayesian_lm_error_test` — tests $H\_0: \lambda = 0$ (no spatial error)
* `bayesian_lm_wx_test` — tests $H\_0: \gamma = 0$ (no WX, from SAR null)
* `bayesian_lm_sdm_joint_test` — joint test for SDM ($H\_0: \rho = 0$ and $\gamma = 0$)
* `bayesian_lm_slx_error_joint_test` — joint test for SDEM ($H\_0: \lambda = 0$ and $\gamma = 0$)
* `bayesian_robust_lm_lag_test` / `bayesian_robust_lm_error_test` — Anselin–Florax robust pair (SAR vs SEM)
* `bayesian_robust_lm_lag_sdm_test` / `bayesian_robust_lm_wx_test` / `bayesian_robust_lm_error_sdem_test` — Neyman-orthogonal robust tests
* `bayesian_lm_error_sdm_test` / `bayesian_lm_lag_sdem_test` — SDM/SDEM-aware tests using correctly filtered residuals

In practice, prefer the method API: `model.spatial_diagnostics()` runs the tests wired to that model class, and `model.spatial_diagnostics_decision()` returns a recommended specification.

Each returns a `BayesianLMTestResult` with posterior summary statistics and a Bayesian p-value.


In [ ]:
# `spatial_diagnostics()` runs all tests wired to a model class
# and returns a tidy DataFrame.
ols.spatial_diagnostics()

## Compare Spatial Parameters Across Models

This cell compares posterior means/intervals for the spatial scalar where present:

* `rho` in SAR/SDM
* `lam` in SEM/SDEM


In [ ]:
# Compare spatial parameters across models
spatial_rows = []
for name, model, var in [
    ("SAR", sar, "rho"),
    ("SEM", sem, "lam"),
    ("SDM", sdm, "rho"),
    ("SDEM", sdem, "lam"),
]:
    if var in model.inference_data.posterior:
        summary = az.summary(model.inference_data, var_names=[var], round_to=3)
        summary.insert(0, "model", name)
        spatial_rows.append(summary)

pd.concat(spatial_rows)

## Matrix-Mode API Example (Optional)

The package also supports direct matrix inputs. This is useful if your design matrix is already engineered.


In [ ]:
y = gdf[ycol]
X = gdf[xcols]

sar_matrix = SAR(y=y, X=X, W=W)
sar_matrix.fit(draws=200, tune=200, chains=2, random_seed=7, progressbar=False)
display(sar_matrix.summary(round_to=3))